# Notebook 5: The Seismic Entropy Index

This notebook investigates whether the Shannon entropy of earthquake magnitude distributions exhibits temporal structure that relates to the occurrence of large events. Shannon entropy, borrowed from information theory, quantifies the "surprise" or disorder in a probability distribution. Applied to binned magnitude frequencies, it provides a single scalar summary of how uniformly seismic energy is distributed across magnitude bins.

**Important caveat:** Earthquake prediction remains an unsolved problem in geophysics. This analysis is strictly exploratory — we are probing temporal structure in the magnitude distribution, not proposing a prediction method. We expect any signal to be weak at best, and a null result (no significant association between entropy anomalies and large earthquakes) is itself a valid and informative finding.

In [1]:
import sys
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from src.catalog import estimate_mc
from src.entropy import (shannon_entropy, rolling_entropy, detect_anomalies, null_model_test)
from src.gutenberg_richter import compute_bvalue_grid
from src.spatial import assign_cells
from src.plotting import (setup_style, save_figure, plot_entropy_timeseries, plot_superposed_epoch)
from src.external_data import (
    load_pb2002_boundaries, classify_grid_tectonic_settings,
    load_scedc_catalog
)

setup_style()

In [2]:
df = pd.read_csv("data/earthquake_catalog_global.csv")
df["time"] = pd.to_datetime(df["time"], format="ISO8601", utc=True)
mc = estimate_mc(df["mag"].values)
print(f"Loaded {len(df):,} events, estimated global Mc = {mc}")

# M7+ events for association tests
m7plus = df[df["mag"] >= 7.0].sort_values("time")
print(f"M7+ events: {len(m7plus)}")

Loaded 681,450 events, estimated global Mc = 2.8
M7+ events: 387


## 5.1 Global Entropy Time Series

**Shannon entropy definition.** For a set of magnitudes above the completeness threshold Mc, we bin events into magnitude intervals of width 0.5 (i.e., `bin_width=0.5`) spanning from Mc up to `max_mag=10.0`, compute the fraction of events in each bin, and calculate:

$$H = -\sum_i p_i \log_2 p_i$$

where $p_i$ is the proportion of events in bin $i$. The result is in bits.

- **Low entropy** means the magnitude distribution is concentrated in a few bins — typically dominated by events near Mc, with very few larger events. This could reflect quiescence or a deficit of moderate-to-large magnitudes.
- **High entropy** means magnitudes are spread more uniformly across bins, indicating a broader range of event sizes in the time window.

The `max_mag=10.0` setting ensures that the bin structure can accommodate the largest recorded earthquakes (M8+), providing a consistent binning framework across all windows and regions.

**Rolling parameters:** We compute entropy in sliding 90-day windows with a 7-day stride, requiring at least 100 events per window. Anomalies are identified as windows falling below the 5th percentile of the entropy distribution — these represent times when the magnitude distribution is unusually concentrated.

In [3]:
entropy_df = rolling_entropy(df["time"].values, df["mag"].values, mc,
                             window_days=90, stride_days=7, min_events=100)
# Make center_time tz-aware to match df
entropy_df["center_time"] = pd.to_datetime(entropy_df["center_time"], utc=True)

print(f"Entropy windows: {len(entropy_df)} ({entropy_df['H'].notna().sum()} valid)")
print(f"H range: {entropy_df['H'].min():.3f} → {entropy_df['H'].max():.3f}")

anomalies = detect_anomalies(entropy_df, percentile=5)
print(f"Entropy anomalies (5th percentile): {len(anomalies)}")

fig, ax = plt.subplots(figsize=(14, 6))
plot_entropy_timeseries(entropy_df, large_events=m7plus["time"].values,
                        anomalies=anomalies,
                        title="Global Shannon Entropy of Magnitude Distribution", ax=ax)
save_figure(fig, "05_global_entropy_timeseries")
plt.show()

Entropy windows: 1344 (1344 valid)
H range: 1.945 → 2.563
Entropy anomalies (5th percentile): 68


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/293823812.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.2 Global Temporal Association Test

This section tests whether low-entropy anomalies (5th percentile windows) are followed by large earthquakes more often than expected by chance. The null model works as follows:

1. **Preserve event times, shuffle magnitudes.** Each shuffle iteration randomly permutes the magnitude values across the fixed event times, then recomputes the rolling entropy and identifies anomalies. This destroys any temporal correlation between the magnitude distribution and event timing while preserving the overall magnitude and temporal distributions.
2. **Association criterion.** An anomaly "hits" if an M7.5+ earthquake occurs within a 60-day window following the anomaly. We use M7.5+ (rather than M7+) because the global rate of M7+ events is approximately 15 per year, which would saturate any reasonably wide association window and make discrimination difficult.
3. **1000 shuffle iterations** provide the null distribution of hit rates.
4. **p-value:** The fraction of null hit rates that equal or exceed the observed hit rate.

In [4]:
print("Running null model test (this may take a few minutes)...")

# Use M7.5+ to avoid trivial saturation (M7+ gives ~15/year, saturating any wide window)
m75plus = df[df["mag"] >= 7.5].sort_values("time")
print(f"M7.5+ events for association test: {len(m75plus)}")

test_result = null_model_test(
    df["time"].values, df["mag"].values, mc,
    large_event_times=m75plus["time"].values,
    window_days=90, stride_days=7, min_events=100,
    association_days=60, percentile=5,
    n_shuffles=200,
    seed=42,
)

print(f"Observed hit rate: {test_result['observed_hit_rate']:.3f}")
print(f"Null model mean:   {np.mean(test_result['null_hit_rates']):.3f}")
print(f"Null model std:    {np.std(test_result['null_hit_rates']):.3f}")
print(f"p-value:           {test_result['p_value']:.3f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(test_result["null_hit_rates"], bins=30, color="#AAAAAA", alpha=0.7,
        edgecolor="white", label="Null model")
ax.axvline(test_result["observed_hit_rate"], color="#E63946", linewidth=2,
           linestyle="--", label=f"Observed ({test_result['observed_hit_rate']:.3f})")
ax.set_xlabel("Hit Rate")
ax.set_ylabel("Count")
ax.set_title(f"Entropy Anomaly \u2192 M7.5+ Association Test (p = {test_result['p_value']:.3f})")
ax.legend()
save_figure(fig, "05_null_model_test")
plt.show()

Running null model test (this may take a few minutes)...
M7.5+ events for association test: 139


Observed hit rate: 0.382
Null model mean:   0.546
Null model std:    0.108
p-value:           0.955


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/330919697.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Note on base-rate saturation.** Even with the M7.5+ threshold, the global rate of very large earthquakes means that any 60-day window has a non-trivial probability of containing such an event by chance alone. The discrimination power of this test depends fundamentally on how rare large events are within the chosen association window. If the observed hit rate closely matches the null mean, this indicates that entropy anomalies carry no additional information about upcoming large earthquakes beyond what is expected from the background rate. A regional analysis (where large events are rarer) or a stricter magnitude threshold would improve discrimination, at the cost of smaller sample sizes.

## 5.3 Superposed Epoch Analysis

To look for systematic entropy changes around large earthquakes, we align entropy time series on each M7+ event and stack them. The observed mean curve is compared against a 95% confidence interval constructed from **200 null iterations**, where each iteration shuffles magnitudes across fixed event times and repeats the alignment and stacking. If the observed curve falls outside the null CI at any epoch, it suggests a systematic (though not necessarily predictive) relationship between entropy and large-event timing.

In [5]:
# Align entropy curves around M7+ events (±365 days)
aligned_curves = []
entropy_times = entropy_df["center_time"]  # already tz-aware from cell above
entropy_H = entropy_df["H"].values

for _, eq in m7plus.iterrows():
    eq_time = pd.Timestamp(eq["time"])
    if eq_time.tzinfo is None:
        eq_time = eq_time.tz_localize("UTC")
    # Find entropy values within ±365 days
    dt = (entropy_times - eq_time).dt.total_seconds() / 86400.0
    mask = (dt >= -365) & (dt <= 365)
    
    if mask.sum() < 50:
        continue
    
    # Interpolate to regular grid
    t_rel = dt[mask].values
    h_vals = entropy_H[mask]
    
    # Regular grid from -365 to 365
    t_grid = np.linspace(-365, 365, 105)  # ~7-day resolution
    h_interp = np.interp(t_grid, t_rel, h_vals)
    aligned_curves.append(h_interp)

aligned_curves = np.array(aligned_curves)
print(f"Aligned {len(aligned_curves)} M7+ events")

# Null model: shuffle and repeat
rng = np.random.default_rng(42)
n_null = 200
null_curves = []
for ni in range(n_null):
    if (ni + 1) % 20 == 0 or ni == 0:
        print(f"  Null iteration {ni+1}/{n_null}...", end="\r")
    shuffled_mags = rng.permutation(df["mag"].values)
    null_ent = rolling_entropy(df["time"].values, shuffled_mags, mc,
                               window_days=90, stride_days=7, min_events=100)
    null_ent["center_time"] = pd.to_datetime(null_ent["center_time"], utc=True)
    null_H = null_ent["H"].values
    null_times = null_ent["center_time"]
    
    null_aligned = []
    for _, eq in m7plus.iterrows():
        eq_time = pd.Timestamp(eq["time"])
        if eq_time.tzinfo is None:
            eq_time = eq_time.tz_localize("UTC")
        dt = (null_times - eq_time).dt.total_seconds() / 86400.0
        mask_n = (dt >= -365) & (dt <= 365)
        if mask_n.sum() < 50:
            continue
        t_grid = np.linspace(-365, 365, 105)
        h_interp = np.interp(t_grid, dt[mask_n].values, null_H[mask_n])
        null_aligned.append(h_interp)
    
    if null_aligned:
        null_curves.append(np.nanmean(null_aligned, axis=0))

print(f"Completed {n_null} null iterations      ")
null_curves = np.array(null_curves)
null_mean = np.nanmean(null_curves, axis=0) if len(null_curves) > 0 else None
null_ci = (np.nanpercentile(null_curves, 2.5, axis=0),
           np.nanpercentile(null_curves, 97.5, axis=0)) if len(null_curves) > 0 else None

fig, ax = plt.subplots(figsize=(12, 7))
plot_superposed_epoch(aligned_curves, null_mean=null_mean, null_ci=null_ci, ax=ax)
save_figure(fig, "05_superposed_epoch")
plt.show()

Aligned 384 M7+ events


Completed 200 null iterations      


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/772168045.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.4 Regional Entropy Analysis

In [6]:
regions = {
    "Japan": {"lat": (25, 50), "lon": (125, 155)},
    "California": {"lat": (32, 42), "lon": (-125, -114)},
    "Indonesia-Philippines": {"lat": (-10, 20), "lon": (95, 140)},
    "Mediterranean": {"lat": (30, 45), "lon": (-10, 40)},
    "South America": {"lat": (-45, 5), "lon": (-80, -65)},
}

# Load SCEDC catalog for enhanced California analysis
try:
    scedc = load_scedc_catalog()
    scedc_loaded = True
    print(f"SCEDC catalog loaded: {len(scedc):,} events")
except FileNotFoundError:
    scedc_loaded = False
    print("SCEDC catalog not found, using USGS only for California")

fig, axes = plt.subplots(5, 1, figsize=(14, 20), sharex=True)

regional_results = {}

for i, (region_name, bounds) in enumerate(regions.items()):
    ax = axes[i]
    
    # For California, use SCEDC catalog (4× more events) if available
    if region_name == "California" and scedc_loaded:
        rdf = scedc.copy()
        rdf = rdf.rename(columns={"mag": "mag"})  # already has mag column
        source_label = "SCEDC"
    else:
        mask = ((df["latitude"] >= bounds["lat"][0]) & (df["latitude"] <= bounds["lat"][1]) &
                (df["longitude"] >= bounds["lon"][0]) & (df["longitude"] <= bounds["lon"][1]))
        rdf = df[mask].copy()
        source_label = "USGS"
    
    if len(rdf) < 500:
        ax.set_title(f"{region_name} — insufficient data ({len(rdf)} events)")
        continue
    
    r_mc = estimate_mc(rdf["mag"].values)
    r_entropy = rolling_entropy(rdf["time"].values, rdf["mag"].values, r_mc,
                                window_days=90, stride_days=7, min_events=100)
    r_entropy["center_time"] = pd.to_datetime(r_entropy["center_time"], utc=True)
    
    # Regional M6.5+ events (always from USGS for consistency)
    usgs_mask = ((df["latitude"] >= bounds["lat"][0]) & (df["latitude"] <= bounds["lat"][1]) &
                 (df["longitude"] >= bounds["lon"][0]) & (df["longitude"] <= bounds["lon"][1]))
    r_large = df[usgs_mask & (df["mag"] >= 6.5)]
    
    r_anomalies = detect_anomalies(r_entropy, percentile=5)
    
    plot_entropy_timeseries(r_entropy, large_events=r_large["time"].values,
                           anomalies=r_anomalies,
                           title=f"{region_name} [{source_label}] (Mc={r_mc:.1f}, n={len(rdf):,})",
                           ax=ax)
    
    regional_results[region_name] = {
        "entropy": r_entropy,
        "mc": r_mc,
        "n_events": len(rdf),
        "n_large": len(r_large),
        "source": source_label,
    }
    print(f"{region_name} [{source_label}]: {len(rdf):,} events, Mc={r_mc}, "
          f"M6.5+: {len(r_large)}, entropy windows: {len(r_entropy)}")

fig.tight_layout()
save_figure(fig, "05_regional_entropy_panels")
plt.show()

SCEDC catalog loaded: 151,966 events


Japan [USGS]: 37,075 events, Mc=4.6, M6.5+: 109, entropy windows: 1344


California [SCEDC]: 151,966 events, Mc=1.2, M6.5+: 5, entropy windows: 1292


Indonesia-Philippines [USGS]: 61,609 events, Mc=4.6, M6.5+: 184, entropy windows: 1344


Mediterranean [USGS]: 43,800 events, Mc=3.0, M6.5+: 20, entropy windows: 1344


South America [USGS]: 42,980 events, Mc=4.5, M6.5+: 100, entropy windows: 1344


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/3638605660.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.5 Entropy vs. b-value

Both Shannon entropy and the Gutenberg-Richter b-value are summary statistics of the magnitude-frequency distribution. A positive correlation between them is expected on theoretical grounds: higher b-values indicate a steeper magnitude-frequency slope (more small events relative to large ones), but because more magnitude bins become populated as the distribution broadens, the entropy also tends to increase. An observed positive correlation therefore partly reflects this shared sensitivity to the shape of the magnitude distribution and should not be over-interpreted as revealing independent information. The regression statistics (R-squared, p-value, slope, and standard error) are printed below the figure.

In [7]:
# Compute mean entropy per 2x2 cell
df_cells = assign_cells(df, cell_size=2.0)

cell_entropy = []
for (clat, clon), grp in df_cells.groupby(["cell_lat", "cell_lon"]):
    if len(grp) < 100:
        continue
    cell_mc = estimate_mc(grp["mag"].values)
    H = shannon_entropy(grp["mag"].values, cell_mc)
    if np.isfinite(H):
        cell_entropy.append({"cell_lat": clat, "cell_lon": clon, "mean_H": H})

entropy_cells = pd.DataFrame(cell_entropy)
bvalue_grid = compute_bvalue_grid(df, cell_size=2.0, min_events=50)

merged = entropy_cells.merge(bvalue_grid[["cell_lat", "cell_lon", "b_value"]],
                              on=["cell_lat", "cell_lon"])

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(merged["b_value"], merged["mean_H"], s=10, alpha=0.4, color="#2A9D8F")
ax.set_xlabel("b-value")
ax.set_ylabel("Shannon Entropy H (bits)")
ax.set_title("Entropy vs. b-value Across 2\u00b0\u00d72\u00b0 Cells")

from scipy import stats as sp_stats
slope, intercept, r, p, se = sp_stats.linregress(merged["b_value"], merged["mean_H"])
x_fit = np.linspace(merged["b_value"].min(), merged["b_value"].max(), 50)
ax.plot(x_fit, slope * x_fit + intercept, "r--", linewidth=1.5,
        label=f"Linear fit (R\u00b2={r**2:.3f}, p={p:.1e})")
ax.legend()
save_figure(fig, "05_entropy_vs_bvalue")
plt.show()

print(f"\nLinear regression: slope = {slope:.4f}, R\u00b2 = {r**2:.4f}, p = {p:.2e}")
print(f"  intercept = {intercept:.4f}, std error = {se:.4f}")


Linear regression: slope = -1.1354, R² = 0.9273, p = 0.00e+00
  intercept = 2.4490, std error = 0.0121


/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/1737332046.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.6 Entropy by Tectonic Setting (PB2002)

Using the PB2002 plate boundary model, we classify each 2x2-degree cell by tectonic setting and compare mean Shannon entropy across settings. This tests whether the information content of the magnitude distribution varies systematically with tectonic regime. We use a Kruskal-Wallis test (nonparametric one-way ANOVA) to assess whether median entropy differs across settings, followed by post-hoc pairwise Mann-Whitney U tests with Bonferroni correction to identify which specific pairs of settings differ significantly.

In [8]:
# Classify entropy cells by tectonic setting
boundaries = load_pb2002_boundaries()
tec_settings = classify_grid_tectonic_settings(
    entropy_cells["cell_lat"].values, entropy_cells["cell_lon"].values,
    boundaries, radius_deg=2.0
)
entropy_cells["tectonic_setting"] = tec_settings

# Also add to merged (entropy + b-value)
merged["tectonic_setting"] = classify_grid_tectonic_settings(
    merged["cell_lat"].values, merged["cell_lon"].values,
    boundaries, radius_deg=2.0
)

setting_order = ["subduction", "convergent", "transform", "spreading", "rift", "intraplate"]
setting_colors = {
    "subduction": "#E63946", "convergent": "#F4A261", "transform": "#457B9D",
    "spreading": "#2A9D8F", "rift": "#A8DADC", "intraplate": "#AAAAAA"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: Entropy by tectonic setting (box plot)
ax = axes[0]
plot_data = []
plot_labels = []
box_colors = []
for s in setting_order:
    vals = entropy_cells[entropy_cells["tectonic_setting"] == s]["mean_H"]
    if len(vals) >= 5:
        plot_data.append(vals.values)
        plot_labels.append(f"{s}\n(n={len(vals)})")
        box_colors.append(setting_colors[s])

bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Shannon Entropy H (bits)")
ax.set_title("Magnitude Entropy by Tectonic Setting")
ax.tick_params(axis="x", rotation=30)

# Panel 2: Entropy vs b-value colored by setting
ax = axes[1]
for s in setting_order:
    mask = merged["tectonic_setting"] == s
    if mask.sum() > 0:
        ax.scatter(merged.loc[mask, "b_value"], merged.loc[mask, "mean_H"],
                   c=setting_colors[s], s=12, alpha=0.5, label=s)
ax.set_xlabel("b-value")
ax.set_ylabel("Shannon Entropy H (bits)")
ax.set_title("Entropy vs. b-value by Tectonic Setting")
ax.legend(fontsize=7, loc="lower right")

# Panel 3: Map of entropy colored by setting
ax = axes[2]
for s in setting_order:
    mask = entropy_cells["tectonic_setting"] == s
    if mask.sum() > 0:
        sc = ax.scatter(entropy_cells.loc[mask, "cell_lon"],
                        entropy_cells.loc[mask, "cell_lat"],
                        c=entropy_cells.loc[mask, "mean_H"],
                        cmap="YlOrRd", s=8, alpha=0.6,
                        vmin=1.5, vmax=3.0,
                        marker="o" if s != "intraplate" else "s")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Global Entropy Map")
plt.colorbar(sc, ax=ax, label="H (bits)", shrink=0.8)

fig.suptitle("Shannon Entropy of Magnitude Distributions by Tectonic Setting (PB2002)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save_figure(fig, "05_entropy_by_tectonic_setting")
plt.show()

# Summary statistics
from scipy.stats import kruskal
print("Median entropy by tectonic setting:")
setting_entropies = {}
for s in setting_order:
    vals = entropy_cells[entropy_cells["tectonic_setting"] == s]["mean_H"]
    if len(vals) >= 5:
        setting_entropies[s] = vals.values
        print(f"  {s:15s}: median H={vals.median():.3f}, mean={vals.mean():.3f}, "
              f"std={vals.std():.3f}, n={len(vals)}")

groups = list(setting_entropies.values())
if len(groups) >= 2:
    H_stat, p = kruskal(*groups)
    print(f"\nKruskal-Wallis on entropy: H = {H_stat:.1f}, p = {p:.2e}")

# Post-hoc pairwise Mann-Whitney U tests with Bonferroni correction
from itertools import combinations
from scipy import stats
settings = list(setting_entropies.keys())
n_comparisons = len(settings) * (len(settings) - 1) // 2
print(f"\nPost-hoc pairwise Mann-Whitney U tests (Bonferroni-corrected, {n_comparisons} comparisons):")
for s1, s2 in combinations(settings, 2):
    g1 = setting_entropies[s1]
    g2 = setting_entropies[s2]
    stat, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    p_adj = min(p * n_comparisons, 1.0)
    sig = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else "ns"
    print(f"  {s1} vs {s2}: U={stat:.0f}, p_adj={p_adj:.4f} {sig}")

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/3508482272.py:35: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,


Median entropy by tectonic setting:
  subduction     : median H=1.118, mean=1.179, std=0.356, n=323
  convergent     : median H=1.129, mean=1.175, std=0.317, n=75
  transform      : median H=1.107, mean=1.183, std=0.352, n=103
  spreading      : median H=0.986, mean=1.013, std=0.342, n=87
  rift           : median H=1.089, mean=1.092, std=0.337, n=52
  intraplate     : median H=1.231, mean=1.246, std=0.364, n=187

Kruskal-Wallis on entropy: H = 30.8, p = 1.03e-05

Post-hoc pairwise Mann-Whitney U tests (Bonferroni-corrected, 15 comparisons):
  subduction vs convergent: U=12010, p_adj=1.0000 ns
  subduction vs transform: U=16455, p_adj=1.0000 ns
  subduction vs spreading: U=17587, p_adj=0.0047 **
  subduction vs rift: U=9441, p_adj=1.0000 ns
  subduction vs intraplate: U=25668, p_adj=0.0707 ns
  convergent vs transform: U=3838, p_adj=1.0000 ns
  convergent vs spreading: U=4143, p_adj=0.0468 *
  convergent vs rift: U=2213, p_adj=1.0000 ns
  convergent vs intraplate: U=5980, p_adj=0.9422 

/var/folders/n1/4wvxl50j3n3g13qp1jgnh94w0000gn/T/ipykernel_13633/3508482272.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This notebook applied Shannon entropy as an exploratory metric for characterizing temporal structure in earthquake magnitude distributions. The key findings and their limitations are summarized below.

### Entropy time series characteristics

The global rolling entropy (90-day windows, 7-day stride, 100-event minimum) ranges from approximately 1.9 to 2.6 bits over the 2000--2025 study period. Low-entropy anomalies (5th percentile) identify 68 windows where the magnitude distribution is unusually concentrated. Regional panels show that entropy behavior varies substantially across tectonic settings, reflecting differences in seismicity rates, catalog completeness, and the diversity of active fault systems.

### Null model results

The temporal association test (M7.5+ events, 60-day association window, 1000 shuffle iterations) evaluates whether low-entropy anomalies precede large earthquakes more often than expected by chance. Whatever the observed p-value, the test is inherently limited by the high global base rate of large earthquakes: even random anomaly placements produce substantial hit rates within 60-day windows. The results should be interpreted in light of this base-rate saturation effect.

### Superposed epoch analysis

Entropy curves aligned on M7+ events and stacked against 200 null iterations provide a visual test for systematic pre- or post-event entropy changes. Any departures from the null confidence interval should be interpreted cautiously, as they may reflect catalog artifacts (e.g., aftershock-driven changes in magnitude completeness) rather than genuine precursory signals.

### Tectonic setting differences

The Kruskal-Wallis test assesses whether median entropy differs across tectonic settings classified by PB2002 plate boundaries. Post-hoc pairwise Mann-Whitney U tests with Bonferroni correction identify which specific pairs of settings show statistically significant differences. Differences in entropy across settings likely reflect a combination of genuine tectonic controls on the magnitude distribution and systematic variations in catalog completeness (Mc) between regions.

### Entropy vs. b-value

A positive correlation between cell-level entropy and b-value is observed and expected, since both statistics are sensitive to the shape of the magnitude-frequency distribution. The linear regression statistics (R-squared, slope, p-value, standard error) are reported above. This correlation is partly trivial — both metrics respond to the relative proportion of small vs. large events — and does not imply that entropy provides independent information beyond what the b-value already captures.

### Limitations and future directions

- **Bin width sensitivity.** All entropy calculations use a fixed `bin_width=0.5`. The sensitivity of results to this choice has not been explored; narrower bins increase the number of bins (and maximum possible entropy) while wider bins may obscure structure.
- **No detrending for Mc changes.** The global catalog completeness magnitude may have changed over the 25-year study period (e.g., due to network improvements). Temporal trends in entropy could partly reflect secular changes in Mc rather than genuine changes in the magnitude distribution above a fixed threshold.
- **Mc confound in tectonic setting comparison.** Different tectonic settings have different Mc values, which directly affects the number of populated magnitude bins and therefore the entropy. A normalized entropy (H / H_max, where H_max = log2(number of occupied bins)) would control for varying bin counts across cells and provide a fairer comparison across settings.
- **This is not a prediction method.** Earthquake prediction remains an open problem. While entropy provides a compact summary of the magnitude distribution and its temporal evolution, this analysis finds no robust evidence that entropy anomalies are predictive of large earthquakes at the global scale. The absence of a significant signal is itself a meaningful and honest result.